# C2 Simulation of an 2D airfoil performance

## Introduction

We are going to simulate from scratch a well known a 2D airfoil. We 
will study the [NACA 65(1)-212 airfoil](https://en.wikipedia.org/wiki/NACA_airfoil#6-series), 
typically used for axial compressors and fans.

We will use the online platform [onShape](https://en.wikipedia.org/wiki/Onshape), a 3D CAD web-based software, which integrates very smoothly with Simscale. In this
way the work flow will be contained on the cloud, and we will not depend on local installed software or computer specifications.  OnShape is not free
but there is an [educational plan](https://www.onshape.com/en/education/)





## Learning objectives

1. Learn to generate a 3D airfoil solid in **onShape** using coordinate data (CSV).
2. Understand the concept of **External Aerodynamics** and size the computational domain.
3. Configure local **Mesh Refinement** and boundary layer around the airfoil
4. Run a **simulation of aerodynamics** in an airfoil with several incidence angles.
5. Compute **aerodynamic coefficients** from the outcome of the simulations


## Previous tasks (about 2 hours)

- [ ] Create and educational account in Onshape 
- [ ] Become familiar with the platform by watching the video for [Onshape essentials webinar](https://learn.onshape.com/learn/video/onshape-essentials)
- [ ] (Optional) Take some tutorial or course from the Learning Center. For instance this [Fundamental course](https://learn.onshape.com/courses/introduction-to-part-studios)



## Making the geometry for the NACA 65(1)-212 airfoil

1. In you dashboard, create a new folder for this subject. For instance, `DFA`, or `Axial Fan`-
2. Make a new Document, named `NACA 65(1)-212`

![create.png](./images/C2_Create.png)

3. Import the CSV file for the airfoil. It is [here](./data/NACA65_1_212.csv). This imported file will be put in a tab of the project.

![import](./images/C2_import.png)

4. Add a `custom feature`. Look for the "Routing curve - Control Point Curve".

![Custom Feature](./images/C2_custom_feature.png)

5. Use this Custom Feature with the imported file (1). Click on "Proceed" (2) and the green button (3).

![RoutingCurve](./images/C2_RoutingCurve.png)

You will get something similar to that

![NACA_curve](./images/C2_Naca_curve.png)

6. `Transform` the curve to scale it for a 0.3 m chord.

![Trasnform](./images/C2_Transform.png)
![scale](./images/C2_Scale.png)

7. `Fill` the curve

![Fill](./images/C2_Fill.png)
![Filled](./images/C2_Filled.png)

8. And finally `Thicken` it 0.3 m to obtain a solid airfoil.

![Thicken](./images/C2_Thicken.png)
![Solid](./images/C2_Solid.png)

## Simulation of the airfoil



### Meshing

Now we are going to simulate the aerodynamic performance of this airfoil with Simscale

1. Sign-in in Simscale and create a `DFA` folder. Create a `New Project` in this folder. You can name it `NACA 65(1)-212`. The category should be `Machinery and Industrial Equipment`
2. The first think asked for is the `Geometry`. Click on "from Onshape` to connect with your account of the cloud service

![Geometry](./images/C2_CFD_geometry.png)

3. Navigate until the proper `Part` in the project. Import it

![Part](./images/C2_CFD_Part.png)

4. Open the geometry and Create the Simulation. Select an `Incompressible` kind with the default parameters (Turbulence model, steady steate...)

![CreateSimulation](./images/C2_CFD_CreateSimulation.png)

5. The first think will be the generation of the mesh. Click on the `Mesh` element, and select `Hex-dominant parametric` with the discretization shown in this picture. Save it. **Still don't generate** 

![Mesh1](./images/C2_CFD_Mesh1.png)

6. Open the `Background Mesh Box` element in the `Geometry primitives` section. Modify the dimensions according to this picture. In the upstream direction we
   get about 5 times the chord (1.5 m), and 15 times downstream (4.5 m), and 5 times the chord (1.5 m) in the top and bottom directions. In the $z$ direction we keep the airfoil dimensions. Save it.

![BMB](./images/C2_CFD_BMB.png) 

7. Be sure that the `Material Point` falls inside the fluid region (outside the airfoil)

![MP](./images/C2_CFD_MP.png)

8. We want to create a region around the airfoil where the mesh will be more refine. Create it by cliking on the `+` symbol in `Geometry primitives` and select `Cartesian box`. Give the dimensions of the figure and name it `Refinement box`. Save it.

![Refinement box](./images/C2_CFD_RefinementBox.png)

9. We define now the refinements. Click on `+` close to `Refinements` and select `Region refinement`. Raise the level to 4 and switch the selector close to `Refinement box`. Don't forget to save it

![levelBox](./images/C2_CFD_levelRefinementBox.png)

10. Add a `Refinement surface` with level 6 on the airfoil. Save it.

![RefinementDurface](./images/C2_CFD_RefinementSurface.png)

11. Add an `Inflate Boundary Layer` on the airfol surface with the parameters of the picture. Save it.

![BoundaryLayer](./images/C2_CFD_BoundaryLayer.png)

12. Back to the `Mesh`, it can now be generated.

![GenerateMesh](./images/C2_CFD_GenerateMesh.png)


It will take a while. In the meantime, read the [Simscale documentation about hex-dominant meshes](https://www.simscale.com/docs/simulation-setup/meshing/hex-dominant/).

After about 5-10 minutes, the mesh, with about 2 Million cells, will be ready for the simulation.

![Mesh Global](./images/C2_CFD_MeshGlobal.png)
![Mesh Detail](./images/C2_CFD_MeshDetail.png)

Since we have defined the mesh before the simulation parameters, the program propose to use the mesh as domain. Click on the proposal, and accept the reassignments of properties. Save it.

![Mesh as Domain](./images/C2_CFD_MeshAsDomain.png)

### Setup of the simulation

We want to compute the aerodynamic force and force coefficients fo this airfoil for several incidence angle. These will be the simulation data:

- Air velocity: $U = 30 \,\text{m/s}$
- Chord length: $c = 0.3\,\text{m}$
- Incidence angles: $\alpha = 0^{\circ}$, $4^{\circ}$ and $8^{\circ}$
  

1. Define the material (fluid). It will be `air`. Keep the default properties. We annotate it for future use in the Notebook 

In [24]:
rho = 1.196  # density of air in kg/m^3
nu = 1.529e-5  # kinematic viscosity of air in m^2/s

2. For the initial condition, we keep $P = 0\,\text{Pa}$, but velocity will depend on the incidence angle (we modify the air direction, not the mesh). Also, once the forces will be computed, we will need to decompose in the incident velocity direction (drag) and its normal (lift). We define three python functions that:
   - Compute Reynolds number
   - Decompose velocity in $x$ and $y$ directions
   - Decompose forces in velocity and its normal directions. Also computes aerodynamic coefficients  

In [25]:
import numpy as np

def ReynoldsNumber(U, c, nu):
    """ 
    Return Reynolds number given the velocity, chord length and kinematic viscosity
    """
    Re = U * c / nu
    print(f"Reynolds number: Re = {Re:.2e}")
    return Re

def velocityComponents(U,alpha):
    """ 
    Return velocity components given the magnitude and the incident angle
    """
    alpha = np.radians(alpha)
    u = U * np.cos(alpha)
    v = U * np.sin(alpha)
    print(f"Velocity components: u = {u:.2f} m/s, v = {v:.2f} m/s")
    return u, v

def LiftDragCoefficients(Fx,Fy,U,rho,c,b,alpha):
    """ 
    Return lift and drag coefficients given the forces, velocity and incident angle
    """
    alpha = np.radians(alpha)
    L = Fy * np.cos(alpha) - Fx * np.sin(alpha)
    D = Fx * np.cos(alpha) + Fy * np.sin(alpha)
    Cl = 2 * L / (rho * U**2 * c * b)
    Cd = 2 * D / (rho * U**2 * c * b)
    print(f"Lift: L = {L:.2f} N, Drag: D = {D:.2f} N")
    print(f"Lift coefficient: Cl = {Cl:.4f}, Drag coefficient: Cd = {Cd:.4f}")
    return Cl, Cd

Define the initial velocity for $\alpha = 0^\circ$, $U_x = 30 \, \text{m/s}$ and $U_y = 0 \, \text{m/s}$. Leave $k$ and $\omega$ (turbulence variables) with ts default values

3. We define the boundary conditions
   1. Define a `Custom - Freestream` boundary condition for the external surfaces in $x$ and $y$ directions.

![Free Stream](./images/C2_CFD_FreeStream.png)

   2. Define a `Wall` boundary condition on the airfoil. Rename it `Airfoil`. 

![Wall](./images/C2_CFD_Wall.png)

   3. Define a `Symmetry` boundary conditions for the front a back surfaces.

![Symmetry](./images/C2_CFD_symmetry.png)  

4. Define the monitor for aerodynamic forces in `Result control - Forces and Moments`. Assign the airfoil surface.

![Forces and Moments](./images/C2_CFD_ForcesMoments.png)

### Running the simulation

Create a nuw raun, named `alpha_0`

![Run alpha 0](./images/C2_CFD_Run_alpha0.png)

In [ ]:
U = 30  # velocity magnitude in m/s
c = 0.3  # chord length in m
b = 0.3  # span in m


ReynoldsNumber(U=U, c=c, nu=nu);

Reynolds number: Re = 5.89e+05


### For $\alpha = 0^\circ$

In [26]:
alpha = 0  # incident angle in degrees
velocityComponents(U=U, alpha=alpha);

Velocity components: u = 30.00 m/s, v = 0.00 m/s


In [27]:
Fx = 0.56  # force in x direction in N
Fy = 6.63  # force in y direction in N

LiftDragCoefficients(Fx=Fx, Fy=Fy, U=U, rho=rho, c=c, b=b, alpha=alpha);

Lift: L = 6.63 N, Drag: D = 0.56 N
Lift coefficient: Cl = 0.1369, Drag coefficient: Cd = 0.0116


### For $\alpha = 4^\circ$

In [28]:
alpha = 4  # incident angle in degrees
velocityComponents(U=U, alpha=alpha);

Velocity components: u = 29.93 m/s, v = 2.09 m/s


In [30]:
Fx = -0.82  # force in x direction in N
Fy = 25.03  # force in y direction in N
LiftDragCoefficients(Fx=Fx, Fy=Fy, U=U, rho=rho, c=c, b=b, alpha=alpha);

Lift: L = 24.90 N, Drag: D = 2.67 N
Lift coefficient: Cl = 0.5141, Drag coefficient: Cd = 0.0552


### For $\alpha = 8^\circ$

In [29]:
alpha = 8  # incident angle in degrees
velocityComponents(U=U, alpha=alpha);

Velocity components: u = 29.71 m/s, v = 4.18 m/s


In [31]:
Fx = -3.69  # force in x direction in N
Fy = 39.16  # force in y direction in N
LiftDragCoefficients(Fx=Fx, Fy=Fy, U=U, rho=rho, c=c, b=b, alpha=alpha);

Lift: L = 39.29 N, Drag: D = 1.80 N
Lift coefficient: Cl = 0.8112, Drag coefficient: Cd = 0.0371
